# 🏗️ Day 6 — NLQ→SQL Workflow Agent on Google Cloud

## Building a Production-Grade Natural Language to SQL Agent with LangGraph, LangChain & Google ADK

---

**Program:** Generative AI Architect Program — Batch 7  
**Provider:** Blue Data Consulting | April 2026  
**Platform:** Google Cloud (BigQuery, Vertex AI, Gemini)  
**Frameworks:** LangGraph, LangChain, Google Agent Development Kit (ADK)

---

### 📋 What You'll Build

In this notebook, you will build a complete **NLQ→SQL Workflow Agent** that:

1. Accepts natural language questions from users
2. Retrieves relevant schema metadata from BigQuery
3. Generates syntactically correct BigQuery SQL using Gemini
4. Validates the SQL through multiple safety gates (dry-run, allow-list, row-limit)
5. Presents the query for human confirmation before execution
6. Executes the query and formats results
7. Self-corrects on errors using a ReAct-style agent loop

We implement this **three ways**:
- **Part A** — LangChain + LangGraph (stateful graph-based orchestration)
- **Part B** — Google ADK (Google's native agent framework)
- **Part C** — Comparison and when to use which

### 📦 Architecture Overview

```
┌──────────────┐     ┌──────────────────┐     ┌────────────────┐
│  User Query  │────▶│  Intent Router   │────▶│ Schema Retriever│
│  (NL Text)   │     │  (classify)      │     │ (BQ metadata)  │
└──────────────┘     └──────────────────┘     └───────┬────────┘
                                                       │
                     ┌──────────────────┐              ▼
                     │  Human Confirm   │◀────┌────────────────┐
                     │  (explain + cost)│     │ SQL Generator  │
                     └────────┬─────────┘     │ (Gemini LLM)  │
                              │               └────────────────┘
                              ▼
                     ┌──────────────────┐     ┌────────────────┐
                     │  SQL Validator   │────▶│  BQ Executor   │
                     │  (dry-run, etc.) │     │  (run + format)│
                     └──────────────────┘     └───────┬────────┘
                                                       │
                              ┌─────────────────────────┘
                              ▼
                     ┌──────────────────┐
                     │  Self-Correction │──── (retry up to 3x)
                     │  (error → fix)   │
                     └──────────────────┘
```


## 📑 Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 0 | [Environment Setup](#0-environment-setup) | Install packages, configure GCP credentials |
| 1 | [BigQuery Setup](#1-bigquery-dataset-setup) | Create dataset, load sample data, add semantic descriptions |
| 2 | [Schema Retriever](#2-schema-retriever) | Build metadata retrieval from INFORMATION_SCHEMA |
| 3 | [SQL Generator](#3-sql-generator-with-gemini) | Prompt engineering + Gemini SQL generation |
| 4 | [Validation Pipeline](#4-sql-validation-pipeline) | Allow-list, dry-run, row-limit, policy checks |
| 5 | [LangGraph Agent](#5-langgraph-nlqsql-agent) | Full stateful agent with LangGraph |
| 6 | [Google ADK Agent](#6-google-adk-nlqsql-agent) | Same agent using Google ADK |
| 7 | [Human-in-the-Loop](#7-human-in-the-loop) | Confirmation step before execution |
| 8 | [Self-Correction](#8-self-correction-loop) | Error handling and retry logic |
| 9 | [Caching & Lineage](#9-caching--lineage-logging) | Result/plan caching, audit trail |
| 10 | [Evaluation](#10-evaluation--testing) | Test with sample queries, measure accuracy |
| 11 | [Comparison](#11-langgraph-vs-adk-comparison) | When to use LangGraph vs ADK |


---
<a id="0-environment-setup"></a>
## 0. Environment Setup

### 0.1 Install Required Packages

We need the following:
- **google-cloud-bigquery** — BigQuery client
- **google-cloud-aiplatform** — Vertex AI / Gemini SDK
- **langchain-google-genai** — LangChain ↔ Gemini integration
- **langchain-google-vertexai** — LangChain ↔ Vertex AI integration
- **langgraph** — Stateful graph-based agent orchestration
- **google-adk** — Google Agent Development Kit
- **db-dtypes** — BigQuery data type support for pandas


In [1]:
# ============================================================
# 0.1 — Install Dependencies
# ============================================================
# Run this cell once. Restart the kernel after installation.

%pip install --quiet --upgrade     google-cloud-bigquery     google-cloud-aiplatform     langchain-core     langchain-google-genai     langchain-google-vertexai     langchain-community     langgraph     google-adk     db-dtypes     pandas     tabulate

Note: you may need to restart the kernel to use updated packages.


### 0.2 Configure GCP Project & Authentication

> **Important:** Replace the placeholder values below with your actual GCP project details.  
> If running on **Vertex AI Workbench** or **Colab Enterprise**, authentication is automatic.  
> If running locally, ensure you've run `gcloud auth application-default login`.


In [2]:
# ============================================================
# 0.2 — GCP Configuration
# ============================================================
import os

# ── Replace these with your actual values ──
PROJECT_ID   = "bdc-trainings"     # e.g., "bdc-trainings"
LOCATION     = "us-central1"              # Vertex AI region
BQ_DATASET   = "nlq_sql_lab"             # BigQuery dataset name
BQ_LOCATION  = "US"                       # BigQuery dataset location

# ── Set environment variables ──
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

print(f"✅ Project: {PROJECT_ID}")
print(f"✅ Location: {LOCATION}")
print(f"✅ Dataset: {BQ_DATASET}")

✅ Project: bdc-trainings
✅ Location: us-central1
✅ Dataset: nlq_sql_lab


In [3]:
# ============================================================
# 0.3 — Authenticate (Colab / Workbench)
# ============================================================
# Uncomment the block below if running in Colab:

# from google.colab import auth
# auth.authenticate_user()

# For Vertex AI Workbench / Cloud Shell, auth is automatic.
# For local dev, ensure: gcloud auth application-default login

from google.cloud import bigquery
import google.cloud.aiplatform as aiplatform

# Initialize Vertex AI
aiplatform.init(project=PROJECT_ID, location=LOCATION)

# Initialize BigQuery client
bq_client = bigquery.Client(project=PROJECT_ID)

print("✅ BigQuery client initialized")
print("✅ Vertex AI initialized")

✅ BigQuery client initialized
✅ Vertex AI initialized


---
<a id="1-bigquery-dataset-setup"></a>
## 1. BigQuery Dataset Setup

We'll create a sample e-commerce dataset with three tables. This simulates a real enterprise warehouse that our NLQ agent will query.

### 1.1 Create Dataset & Tables


In [4]:
# ============================================================
# 1.1 — Create BigQuery Dataset
# ============================================================
from google.cloud import bigquery

dataset_ref = bigquery.DatasetReference(PROJECT_ID, BQ_DATASET)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = BQ_LOCATION
dataset.description = "Sample e-commerce dataset for NLQ→SQL lab"

try:
    bq_client.create_dataset(dataset, exists_ok=True)
    print(f"✅ Dataset '{BQ_DATASET}' created/verified")
except Exception as e:
    print(f"Dataset creation: {e}")

✅ Dataset 'nlq_sql_lab' created/verified


In [5]:
# ============================================================
# 1.2 — Create Tables with Sample Data
# ============================================================

# ── Table 1: orders ──
ORDERS_DDL = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{BQ_DATASET}.orders` (
    order_id        STRING      OPTIONS(description='Unique order identifier (UUID format)'),
    customer_id     STRING      OPTIONS(description='Customer identifier linking to customers table'),
    order_date      DATE        OPTIONS(description='Date when the order was placed (UTC)'),
    status          STRING      OPTIONS(description='Order status: COMPLETED, PENDING, CANCELLED, REFUNDED'),
    total_amount    FLOAT64     OPTIONS(description='Total order amount in USD after discounts, before tax'),
    region          STRING      OPTIONS(description='Geographic sales region: North America, Europe, Asia Pacific, Latin America'),
    channel         STRING      OPTIONS(description='Sales channel: web, mobile, in-store, marketplace')
)
OPTIONS(
    description='Transactional orders table. One row per order. Granularity: order-level. Updated daily via ETL pipeline.'
)
"""

# ── Table 2: customers ──
CUSTOMERS_DDL = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{BQ_DATASET}.customers` (
    customer_id     STRING      OPTIONS(description='Unique customer identifier (UUID format)'),
    name            STRING      OPTIONS(description='Full name of the customer'),
    email           STRING      OPTIONS(description='Customer email address (PII - restrict access)'),
    segment         STRING      OPTIONS(description='Customer segment: Enterprise, SMB, Consumer, Government'),
    country         STRING      OPTIONS(description='ISO 3166-1 alpha-2 country code (e.g., US, GB, DE)'),
    created_at      DATE        OPTIONS(description='Account creation date (UTC)')
)
OPTIONS(
    description='Customer master table. One row per customer. Contains PII - access restricted to authorized roles.'
)
"""

# ── Table 3: products ──
PRODUCTS_DDL = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{BQ_DATASET}.products` (
    product_id      STRING      OPTIONS(description='Unique product SKU identifier'),
    name            STRING      OPTIONS(description='Product display name'),
    category        STRING      OPTIONS(description='Product category: Electronics, Clothing, Home & Garden, Software, Services'),
    price           FLOAT64     OPTIONS(description='Current unit price in USD'),
    cost            FLOAT64     OPTIONS(description='Unit cost/COGS in USD')
)
OPTIONS(
    description='Product catalog table. One row per product. Updated by merchandising team weekly.'
)
"""

# ── Table 4: order_items (join table) ──
ORDER_ITEMS_DDL = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{BQ_DATASET}.order_items` (
    order_id        STRING      OPTIONS(description='Foreign key to orders.order_id'),
    product_id      STRING      OPTIONS(description='Foreign key to products.product_id'),
    quantity        INT64       OPTIONS(description='Number of units ordered'),
    unit_price      FLOAT64     OPTIONS(description='Price per unit at time of order in USD'),
    discount_pct    FLOAT64     OPTIONS(description='Discount percentage applied (0.0 to 1.0 scale)')
)
OPTIONS(
    description='Order line items. One row per product per order. Links orders to products.'
)
"""

# Execute DDL statements
for ddl_name, ddl_sql in [("orders", ORDERS_DDL), ("customers", CUSTOMERS_DDL),
                            ("products", PRODUCTS_DDL), ("order_items", ORDER_ITEMS_DDL)]:
    bq_client.query(ddl_sql).result()
    print(f"  ✅ Table '{ddl_name}' created")

print("\n✅ All tables created with semantic descriptions")

  ✅ Table 'orders' created
  ✅ Table 'customers' created
  ✅ Table 'products' created
  ✅ Table 'order_items' created

✅ All tables created with semantic descriptions


In [ ]:
# ============================================================
# 1.3 — Load Sample Data
# ============================================================

SEED_DATA_SQL = f"""
-- Insert sample customers
INSERT INTO `{PROJECT_ID}.{BQ_DATASET}.customers` VALUES
('C001', 'Acme Corp', 'procurement@acme.com', 'Enterprise', 'US', '2023-01-15'),
('C002', 'TechStart GmbH', 'orders@techstart.de', 'SMB', 'DE', '2023-03-22'),
('C003', 'Jane Smith', 'jane.smith@email.com', 'Consumer', 'US', '2023-06-01'),
('C004', 'GlobalMfg Ltd', 'supply@globalmfg.co.uk', 'Enterprise', 'GB', '2023-02-10'),
('C005', 'Sakura Electronics', 'info@sakura-e.co.jp', 'SMB', 'JP', '2023-07-18'),
('C006', 'Maria Garcia', 'maria.g@email.com', 'Consumer', 'US', '2024-01-05'),
('C007', 'Nordic Solutions AB', 'sales@nordic-sol.se', 'Enterprise', 'SE', '2023-09-30'),
('C008', 'DataFlow Inc', 'admin@dataflow.io', 'SMB', 'US', '2024-03-12');

-- Insert sample products
INSERT INTO `{PROJECT_ID}.{BQ_DATASET}.products` VALUES
('P001', 'Cloud Server Pro', 'Software', 2999.99, 800.00),
('P002', 'Smart Monitor 32"', 'Electronics', 549.99, 220.00),
('P003', 'Ergonomic Keyboard', 'Electronics', 149.99, 45.00),
('P004', 'Office Chair Deluxe', 'Home & Garden', 699.99, 280.00),
('P005', 'AI Analytics Suite', 'Software', 4999.99, 1200.00),
('P006', 'USB-C Hub Pro', 'Electronics', 79.99, 22.00),
('P007', 'Standing Desk', 'Home & Garden', 899.99, 350.00),
('P008', 'Premium Support Plan', 'Services', 1999.99, 500.00);

-- Insert sample orders (2024-2025)
INSERT INTO `{PROJECT_ID}.{BQ_DATASET}.orders` VALUES
('ORD-001', 'C001', '2024-07-15', 'COMPLETED', 15499.95, 'North America', 'web'),
('ORD-002', 'C002', '2024-08-22', 'COMPLETED', 3099.98, 'Europe', 'web'),
('ORD-003', 'C003', '2024-09-10', 'COMPLETED', 229.98, 'North America', 'mobile'),
('ORD-004', 'C004', '2024-10-05', 'COMPLETED', 9999.98, 'Europe', 'marketplace'),
('ORD-005', 'C005', '2024-11-18', 'PENDING', 699.99, 'Asia Pacific', 'web'),
('ORD-006', 'C001', '2024-12-01', 'COMPLETED', 4999.99, 'North America', 'web'),
('ORD-007', 'C006', '2025-01-14', 'COMPLETED', 629.98, 'North America', 'mobile'),
('ORD-008', 'C002', '2025-01-28', 'CANCELLED', 149.99, 'Europe', 'web'),
('ORD-009', 'C007', '2025-02-10', 'COMPLETED', 8999.97, 'Europe', 'web'),
('ORD-010', 'C003', '2025-03-05', 'REFUNDED', 549.99, 'North America', 'web'),
('ORD-011', 'C008', '2025-03-20', 'COMPLETED', 7699.97, 'North America', 'web'),
('ORD-012', 'C004', '2025-04-01', 'COMPLETED', 1999.99, 'Europe', 'in-store'),
('ORD-013', 'C005', '2025-04-15', 'PENDING', 2999.99, 'Asia Pacific', 'web'),
('ORD-014', 'C001', '2025-05-01', 'COMPLETED', 11499.94, 'North America', 'web'),
('ORD-015', 'C006', '2025-05-20', 'COMPLETED', 79.99, 'North America', 'mobile');

-- Insert sample order items
INSERT INTO `{PROJECT_ID}.{BQ_DATASET}.order_items` VALUES
('ORD-001', 'P001', 3, 2999.99, 0.0),  ('ORD-001', 'P005', 1, 4999.99, 0.1),
('ORD-001', 'P008', 1, 1999.99, 0.0),  ('ORD-002', 'P002', 2, 549.99, 0.0),
('ORD-002', 'P003', 1, 149.99, 0.0),   ('ORD-002', 'P006', 5, 79.99, 0.15),
('ORD-003', 'P003', 1, 149.99, 0.0),   ('ORD-003', 'P006', 1, 79.99, 0.0),
('ORD-004', 'P005', 2, 4999.99, 0.0),  ('ORD-005', 'P004', 1, 699.99, 0.0),
('ORD-006', 'P005', 1, 4999.99, 0.0),  ('ORD-007', 'P002', 1, 549.99, 0.0),
('ORD-007', 'P006', 1, 79.99, 0.0),    ('ORD-008', 'P003', 1, 149.99, 0.0),
('ORD-009', 'P001', 3, 2999.99, 0.0),  ('ORD-010', 'P002', 1, 549.99, 0.0),
('ORD-011', 'P001', 1, 2999.99, 0.0),  ('ORD-011', 'P005', 1, 4999.99, 0.1),
('ORD-012', 'P008', 1, 1999.99, 0.0),  ('ORD-013', 'P001', 1, 2999.99, 0.0),
('ORD-014', 'P001', 2, 2999.99, 0.0),  ('ORD-014', 'P005', 1, 4999.99, 0.05),
('ORD-015', 'P006', 1, 79.99, 0.0);
"""

# Execute each INSERT block separately
for statement in SEED_DATA_SQL.strip().split(';'):
    stmt = statement.strip()
    if stmt and stmt.startswith('--'):
        try:
            #print(stmt)
            bq_client.query(stmt).result()
        except Exception as e:
            if "already exists" not in str(e).lower():
                print(f"  ⚠️ {e}")

# Verify row counts
for table in ['customers', 'products', 'orders', 'order_items']:
    q = f"SELECT COUNT(*) as cnt FROM `{PROJECT_ID}.{BQ_DATASET}.{table}`"
    count = list(bq_client.query(q).result())[0].cnt
    print(f"  ✅ {table}: {count} rows")

print("\n✅ Sample data loaded successfully")

-- Insert sample customers
INSERT INTO `bdc-trainings.nlq_sql_lab.customers` VALUES
('C001', 'Acme Corp', 'procurement@acme.com', 'Enterprise', 'US', '2023-01-15'),
('C002', 'TechStart GmbH', 'orders@techstart.de', 'SMB', 'DE', '2023-03-22'),
('C003', 'Jane Smith', 'jane.smith@email.com', 'Consumer', 'US', '2023-06-01'),
('C004', 'GlobalMfg Ltd', 'supply@globalmfg.co.uk', 'Enterprise', 'GB', '2023-02-10'),
('C005', 'Sakura Electronics', 'info@sakura-e.co.jp', 'SMB', 'JP', '2023-07-18'),
('C006', 'Maria Garcia', 'maria.g@email.com', 'Consumer', 'US', '2024-01-05'),
('C007', 'Nordic Solutions AB', 'sales@nordic-sol.se', 'Enterprise', 'SE', '2023-09-30'),
('C008', 'DataFlow Inc', 'admin@dataflow.io', 'SMB', 'US', '2024-03-12')
-- Insert sample products
INSERT INTO `bdc-trainings.nlq_sql_lab.products` VALUES
('P001', 'Cloud Server Pro', 'Software', 2999.99, 800.00),
('P002', 'Smart Monitor 32"', 'Electronics', 549.99, 220.00),
('P003', 'Ergonomic Keyboard', 'Electronics', 149.99, 45.00),
(

In [10]:
SEED_DATA_SQL.strip().split(';')[0].strip().startswith('--')

True

---
<a id="2-schema-retriever"></a>
## 2. Schema Retriever

The schema retriever is the **most critical component** of any NLQ→SQL system. It provides the LLM with the context it needs to generate accurate SQL.

### Why Not Just Dump the Entire Schema?

In enterprise environments, data warehouses have hundreds or thousands of tables. Stuffing all metadata into a single prompt would:
- Exceed context window limits
- Confuse the LLM with irrelevant tables
- Waste tokens and increase cost/latency

Instead, we build a **metadata-aware retrieval pipeline** that fetches only the relevant tables and columns.

### 2.1 Metadata Retrieval Functions


In [12]:
# ============================================================
# 2.1 — Schema Retriever: Core Functions
# ============================================================
from google.cloud import bigquery
from typing import Optional
import json

class SchemaRetriever:
    """
    Retrieves and formats BigQuery schema metadata for LLM consumption.
    
    This class provides:
    - Table-level metadata (descriptions, row counts)
    - Column-level metadata (types, descriptions)
    - Sample rows for context
    - Relationship hints (FK conventions)
    - Formatted output suitable for LLM prompts
    """
    
    # ── Allow-list: only these tables can be queried ──
    ALLOWED_TABLES = {"orders", "customers", "products", "order_items"}
    
    # ── Deny-list: columns the LLM must never include in output ──
    DENIED_COLUMNS = {"email"}  # PII protection
    
    def __init__(self, project_id: str, dataset_id: str, client: bigquery.Client):
        self.project_id = project_id
        self.dataset_id = dataset_id
        self.client = client
        self._cache = {}  # Simple in-memory cache
    
    def get_all_tables_summary(self) -> str:
        """Get a high-level summary of all allowed tables in the dataset."""
        cache_key = "tables_summary"
        if cache_key in self._cache:
            return self._cache[cache_key]
        
        query = f"""
        SELECT
            table_name,
            IFNULL(option_value, 'No description') AS table_description
        FROM `{self.project_id}.{self.dataset_id}.INFORMATION_SCHEMA.TABLE_OPTIONS`
        WHERE option_name = 'description'
        ORDER BY table_name
        """
        rows = list(self.client.query(query).result())
        
        summary_parts = ["## Available Tables\n"]
        for row in rows:
            if row.table_name in self.ALLOWED_TABLES:
                summary_parts.append(
                    f"- **{row.table_name}**: {row.table_description}"
                )
        
        result = "\n".join(summary_parts)
        self._cache[cache_key] = result
        return result
    
    def get_table_schema(self, table_name: str) -> str:
        """
        Get detailed schema for a specific table in CREATE TABLE DDL format.
        This format has been shown to be most effective for LLM SQL generation.
        """
        if table_name not in self.ALLOWED_TABLES:
            return f"ERROR: Table '{table_name}' is not in the allow-list."
        
        cache_key = f"schema_{table_name}"
        if cache_key in self._cache:
            return self._cache[cache_key]
        
        # Get column details
        query = f"""
        SELECT
            c.column_name,
            c.data_type,
            c.is_nullable,
            IFNULL(o.option_value, '') AS column_description
        FROM `{self.project_id}.{self.dataset_id}.INFORMATION_SCHEMA.COLUMNS` c
        LEFT JOIN `{self.project_id}.{self.dataset_id}.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS` o
            ON c.table_name = o.table_name AND c.column_name = o.column_name
        WHERE c.table_name = '{table_name}'
        ORDER BY c.ordinal_position
        """
        rows = list(self.client.query(query).result())
        
        # Get table description
        desc_query = f"""
        SELECT option_value FROM
        `{self.project_id}.{self.dataset_id}.INFORMATION_SCHEMA.TABLE_OPTIONS`
        WHERE table_name = '{table_name}' AND option_name = 'description'
        """
        desc_rows = list(self.client.query(desc_query).result())
        table_desc = desc_rows[0].option_value if desc_rows else "No description"
        
        # Build DDL format (most effective for LLM comprehension)
        ddl_parts = [f"-- {table_desc}"]
        ddl_parts.append(f"CREATE TABLE `{self.dataset_id}.{table_name}` (")
        
        col_lines = []
        for row in rows:
            if row.column_name in self.DENIED_COLUMNS:
                continue  # Skip PII columns
            desc_comment = f"  -- {row.column_description}" if row.column_description else ""
            nullable = "" if row.is_nullable == "YES" else " NOT NULL"
            col_lines.append(
                f"    {row.column_name:<20s} {row.data_type}{nullable}{desc_comment}"
            )
        
        ddl_parts.append(",\n".join(col_lines))
        ddl_parts.append(");")
        
        result = "\n".join(ddl_parts)
        self._cache[cache_key] = result
        return result
    
    def get_sample_rows(self, table_name: str, limit: int = 3) -> str:
        """Get sample rows to help the LLM understand data format and values."""
        if table_name not in self.ALLOWED_TABLES:
            return f"ERROR: Table '{table_name}' is not in the allow-list."
        
        # Exclude denied columns from SELECT
        cols_query = f"""
        SELECT column_name FROM
        `{self.project_id}.{self.dataset_id}.INFORMATION_SCHEMA.COLUMNS`
        WHERE table_name = '{table_name}'
        ORDER BY ordinal_position
        """
        cols = [r.column_name for r in self.client.query(cols_query).result()
                if r.column_name not in self.DENIED_COLUMNS]
        
        col_list = ", ".join(cols)
        query = f"SELECT {col_list} FROM `{self.project_id}.{self.dataset_id}.{table_name}` LIMIT {limit}"
        rows = list(self.client.query(query).result())
        
        if not rows:
            return f"No sample data in {table_name}"
        
        # Format as a readable table
        import pandas as pd
        df = pd.DataFrame([dict(row) for row in rows])
        return f"Sample rows from {table_name}:\n{df.to_string(index=False)}"
    
    def get_relationships(self) -> str:
        """Return known table relationships (FK conventions)."""
        return """## Table Relationships
- orders.customer_id → customers.customer_id (many-to-one)
- order_items.order_id → orders.order_id (many-to-one)
- order_items.product_id → products.product_id (many-to-one)

## Key Business Rules
- Revenue = SUM(order_items.unit_price * order_items.quantity * (1 - order_items.discount_pct))
- Use orders.status = 'COMPLETED' for revenue calculations (exclude PENDING, CANCELLED, REFUNDED)
- Regions: 'North America', 'Europe', 'Asia Pacific', 'Latin America'
- Customer segments: 'Enterprise', 'SMB', 'Consumer', 'Government'
"""
    
    def get_full_context(self, relevant_tables: Optional[list] = None) -> str:
        """
        Build complete context for the LLM prompt.
        If relevant_tables is provided, only include those tables.
        Otherwise, include all allowed tables.
        """
        tables = relevant_tables or list(self.ALLOWED_TABLES)
        tables = [t for t in tables if t in self.ALLOWED_TABLES]
        
        parts = [self.get_all_tables_summary(), "\n"]
        parts.append("## Detailed Schema\n")
        for table in sorted(tables):
            parts.append(self.get_table_schema(table))
            parts.append("")
        parts.append(self.get_relationships())
        
        return "\n".join(parts)

# ── Instantiate ──
schema_retriever = SchemaRetriever(PROJECT_ID, BQ_DATASET, bq_client)
print("✅ SchemaRetriever initialized")

✅ SchemaRetriever initialized


In [13]:
# ============================================================
# 2.2 — Test Schema Retriever
# ============================================================

# View the full context that will be injected into the LLM prompt
context = schema_retriever.get_full_context()
print(context)

BadRequest: 400 Name option_value not found inside o at [6:22]; reason: invalidQuery, location: query, message: Name option_value not found inside o at [6:22]

Location: US
Job ID: 3ff068e7-d3f3-46c5-aa43-4cffc04b8df9


In [ ]:
# ============================================================
# 2.3 — View Sample Data
# ============================================================

for table in ["orders", "customers", "products"]:
    print(schema_retriever.get_sample_rows(table))
    print()

---
<a id="3-sql-generator-with-gemini"></a>
## 3. SQL Generator with Gemini

### 3.1 Prompt Engineering for SQL Generation

This is where the magic happens. Our prompt template follows industry best practices:

1. **Role definition** — Tell Gemini it's a BigQuery SQL expert
2. **Schema injection** — DDL format with descriptions (from our SchemaRetriever)
3. **Output constraints** — SQL only, no markdown, specific BigQuery dialect
4. **Few-shot examples** — Representative Q→SQL pairs
5. **Safety rules** — No SELECT *, must use WHERE on large tables


In [ ]:
# ============================================================
# 3.1 — SQL Generation Prompt Template
# ============================================================

SYSTEM_PROMPT = """You are an expert BigQuery SQL analyst. Your job is to translate
natural language questions into correct, optimized BigQuery Standard SQL queries.

## STRICT RULES — follow these exactly:
1. Return ONLY the SQL query. No explanations, no markdown fences, no comments.
2. Use BigQuery Standard SQL syntax (not Legacy SQL).
3. NEVER use SELECT * — always list specific columns.
4. Always use table aliases for readability (e.g., o for orders, c for customers).
5. For revenue/sales calculations, only include orders with status = 'COMPLETED'.
6. Always include a WHERE clause when filtering is implied by the question.
7. Use fully qualified table names: `{dataset}.table_name`
8. For date comparisons, use DATE functions (EXTRACT, DATE_TRUNC, CURRENT_DATE).
9. If the question is ambiguous, make the most reasonable interpretation and proceed.
10. If the question cannot be answered with the available schema, respond with: UNSUPPORTED_QUERY

## CURRENT DATE CONTEXT
Today's date is: {current_date}
"""

FEW_SHOT_EXAMPLES = """
## Example Questions and SQL

Q: What is total revenue by region?
SQL: SELECT o.region, SUM(o.total_amount) AS total_revenue FROM `{dataset}.orders` o WHERE o.status = 'COMPLETED' GROUP BY o.region ORDER BY total_revenue DESC

Q: How many orders did we get per month in 2025?
SQL: SELECT DATE_TRUNC(o.order_date, MONTH) AS order_month, COUNT(*) AS order_count FROM `{dataset}.orders` o WHERE EXTRACT(YEAR FROM o.order_date) = 2025 GROUP BY order_month ORDER BY order_month

Q: Who are our top 5 customers by total spend?
SQL: SELECT c.customer_id, c.name, c.segment, SUM(o.total_amount) AS total_spend FROM `{dataset}.orders` o JOIN `{dataset}.customers` c ON o.customer_id = c.customer_id WHERE o.status = 'COMPLETED' GROUP BY c.customer_id, c.name, c.segment ORDER BY total_spend DESC LIMIT 5

Q: What is the best-selling product by quantity?
SQL: SELECT p.product_id, p.name, p.category, SUM(oi.quantity) AS total_qty FROM `{dataset}.order_items` oi JOIN `{dataset}.products` p ON oi.product_id = p.product_id JOIN `{dataset}.orders` o ON oi.order_id = o.order_id WHERE o.status = 'COMPLETED' GROUP BY p.product_id, p.name, p.category ORDER BY total_qty DESC LIMIT 10
"""

print("✅ Prompt templates defined")

In [ ]:
# ============================================================
# 3.2 — SQL Generator Class
# ============================================================
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from datetime import date

class SQLGenerator:
    """
    Generates BigQuery SQL from natural language using Gemini.
    
    Uses LangChain's ChatGoogleGenerativeAI for clean integration
    with Google's Gemini models via Vertex AI.
    """
    
    def __init__(self, model_name: str = "gemini-2.5-flash", temperature: float = 0.0):
        self.llm = ChatGoogleGenerativeAI(
            model=model_name,
            temperature=temperature,      # Low temp for deterministic SQL
            max_output_tokens=2048,
            google_api_key=None,          # Uses application default credentials
        )
        self.model_name = model_name
    
    def generate(
        self,
        question: str,
        schema_context: str,
        dataset: str,
    ) -> dict:
        """
        Generate SQL from a natural language question.
        
        Args:
            question: Natural language question from the user
            schema_context: Formatted schema metadata from SchemaRetriever
            dataset: BigQuery dataset name for fully-qualified table references
        
        Returns:
            dict with keys: sql, model, question
        """
        current_date_str = date.today().isoformat()
        
        system = SYSTEM_PROMPT.format(
            dataset=dataset,
            current_date=current_date_str
        )
        
        few_shot = FEW_SHOT_EXAMPLES.format(dataset=dataset)
        
        user_prompt = f"""
{schema_context}

{few_shot}

## User Question
Q: {question}
SQL:"""
        
        messages = [
            SystemMessage(content=system),
            HumanMessage(content=user_prompt),
        ]
        
        response = self.llm.invoke(messages)
        
        # Clean the response — strip markdown fences if the model adds them
        sql = response.content.strip()
        sql = sql.removeprefix("```sql").removeprefix("```").removesuffix("```").strip()
        
        return {
            "sql": sql,
            "model": self.model_name,
            "question": question,
        }

# ── Instantiate ──
sql_generator = SQLGenerator(model_name="gemini-2.5-flash", temperature=0.0)
print("✅ SQLGenerator initialized with Gemini 2.5 Flash")

In [ ]:
# ============================================================
# 3.3 — Test SQL Generation
# ============================================================

# Get schema context
schema_context = schema_retriever.get_full_context()

# Test questions
test_questions = [
    "What is total revenue by region?",
    "Show me the top 3 customers by total spend",
    "How many orders were placed each month in 2025?",
    "What is the average order value by sales channel?",
]

for q in test_questions:
    result = sql_generator.generate(q, schema_context, BQ_DATASET)
    print(f"Q: {q}")
    print(f"SQL: {result['sql']}")
    print("-" * 80)

---
<a id="4-sql-validation-pipeline"></a>
## 4. SQL Validation Pipeline

**Never auto-execute LLM-generated SQL.** Every query must pass through multiple safety gates.

Our validation pipeline has four stages:
1. **Column Allow-List Check** — Ensure all referenced columns exist in our schema
2. **Policy Check** — Block dangerous patterns (SELECT *, CROSS JOIN, etc.)
3. **Dry-Run Gate** — Estimate bytes processed without executing
4. **Row-Limit Injection** — Auto-append LIMIT for safety


In [ ]:
# ============================================================
# 4.1 — SQL Validator
# ============================================================
import re
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class ValidationResult:
    """Result of SQL validation pipeline."""
    is_valid: bool
    sql: str                              # Potentially modified SQL (e.g., LIMIT added)
    original_sql: str
    errors: list = field(default_factory=list)
    warnings: list = field(default_factory=list)
    estimated_bytes: Optional[int] = None
    estimated_cost_usd: Optional[float] = None
    
    def summary(self) -> str:
        status = "✅ VALID" if self.is_valid else "❌ INVALID"
        parts = [f"Status: {status}"]
        if self.estimated_bytes:
            mb = self.estimated_bytes / (1024 * 1024)
            parts.append(f"Estimated scan: {mb:.1f} MB (${self.estimated_cost_usd:.4f} USD)")
        if self.errors:
            parts.append(f"Errors: {'; '.join(self.errors)}")
        if self.warnings:
            parts.append(f"Warnings: {'; '.join(self.warnings)}")
        return " | ".join(parts)


class SQLValidator:
    """
    Multi-stage SQL validation pipeline for production NLQ→SQL systems.
    
    Stages:
    1. Syntax & policy checks (regex-based)
    2. Column/table allow-list validation
    3. BigQuery dry-run (bytes estimate)
    4. Row-limit injection
    """
    
    # BigQuery on-demand pricing: $6.25 per TB
    BQ_COST_PER_TB = 6.25
    
    # Maximum allowed bytes before requiring explicit confirmation
    MAX_BYTES_AUTO = 1 * 1024**3  # 1 GB
    
    # Default row limit for exploratory queries
    DEFAULT_ROW_LIMIT = 1000
    
    # Dangerous SQL patterns to block
    BLOCKED_PATTERNS = [
        (r"\bSELECT\s+\*\b", "SELECT * is not allowed — specify columns explicitly"),
        (r"\bDROP\b", "DROP statements are blocked"),
        (r"\bDELETE\b", "DELETE statements are blocked"),
        (r"\bUPDATE\b", "UPDATE statements are blocked"),
        (r"\bINSERT\b", "INSERT statements are blocked"),
        (r"\bTRUNCATE\b", "TRUNCATE statements are blocked"),
        (r"\bCREATE\b", "CREATE statements are blocked"),
        (r"\bALTER\b", "ALTER statements are blocked"),
        (r"\bMERGE\b", "MERGE statements are blocked"),
        (r"\bCROSS\s+JOIN\b", "CROSS JOIN is blocked for safety"),
    ]
    
    def __init__(
        self,
        project_id: str,
        dataset_id: str,
        client: bigquery.Client,
        allowed_tables: set,
    ):
        self.project_id = project_id
        self.dataset_id = dataset_id
        self.client = client
        self.allowed_tables = allowed_tables
    
    def validate(self, sql: str) -> ValidationResult:
        """Run the full validation pipeline."""
        result = ValidationResult(is_valid=True, sql=sql, original_sql=sql)
        
        # Stage 1: Policy checks
        self._check_policies(sql, result)
        if not result.is_valid:
            return result
        
        # Stage 2: Table allow-list check
        self._check_tables(sql, result)
        if not result.is_valid:
            return result
        
        # Stage 3: Row-limit injection
        self._inject_row_limit(result)
        
        # Stage 4: Dry-run
        self._dry_run(result)
        
        return result
    
    def _check_policies(self, sql: str, result: ValidationResult):
        """Check SQL against blocked patterns."""
        for pattern, message in self.BLOCKED_PATTERNS:
            if re.search(pattern, sql, re.IGNORECASE):
                result.is_valid = False
                result.errors.append(f"POLICY VIOLATION: {message}")
    
    def _check_tables(self, sql: str, result: ValidationResult):
        """Verify all referenced tables are in the allow-list."""
        # Extract table references — look for dataset.table or just table names after FROM/JOIN
        table_pattern = rf"(?:FROM|JOIN)\s+`?(?:{re.escape(self.dataset_id)}\.)?(\w+)`?"
        referenced = set(re.findall(table_pattern, sql, re.IGNORECASE))
        
        unauthorized = referenced - self.allowed_tables
        if unauthorized:
            result.is_valid = False
            result.errors.append(
                f"UNAUTHORIZED TABLES: {', '.join(unauthorized)}. "
                f"Allowed: {', '.join(sorted(self.allowed_tables))}"
            )
    
    def _inject_row_limit(self, result: ValidationResult):
        """Add LIMIT clause if not already present."""
        if not re.search(r"\bLIMIT\b", result.sql, re.IGNORECASE):
            result.sql = result.sql.rstrip(";") + f"\nLIMIT {self.DEFAULT_ROW_LIMIT}"
            result.warnings.append(f"Auto-added LIMIT {self.DEFAULT_ROW_LIMIT}")
    
    def _dry_run(self, result: ValidationResult):
        """Execute BigQuery dry-run to estimate bytes processed."""
        try:
            job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
            query_job = self.client.query(result.sql, job_config=job_config)
            
            result.estimated_bytes = query_job.total_bytes_processed
            result.estimated_cost_usd = (
                result.estimated_bytes / (1024**4) * self.BQ_COST_PER_TB
            )
            
            if result.estimated_bytes > self.MAX_BYTES_AUTO:
                result.warnings.append(
                    f"Query will scan {result.estimated_bytes / 1024**3:.1f} GB — "
                    f"exceeds auto-execute threshold ({self.MAX_BYTES_AUTO / 1024**3:.0f} GB). "
                    f"Requires explicit confirmation."
                )
        except Exception as e:
            result.is_valid = False
            result.errors.append(f"DRY-RUN FAILED: {str(e)}")


# ── Instantiate ──
sql_validator = SQLValidator(
    project_id=PROJECT_ID,
    dataset_id=BQ_DATASET,
    client=bq_client,
    allowed_tables=SchemaRetriever.ALLOWED_TABLES,
)
print("✅ SQLValidator initialized")

In [ ]:
# ============================================================
# 4.2 — Test Validation Pipeline
# ============================================================

# Test 1: Valid query
valid_sql = f"SELECT o.region, SUM(o.total_amount) AS revenue FROM `{BQ_DATASET}.orders` o WHERE o.status = 'COMPLETED' GROUP BY o.region ORDER BY revenue DESC"
result1 = sql_validator.validate(valid_sql)
print("Test 1 (valid query):")
print(f"  {result1.summary()}")
print(f"  Final SQL: {result1.sql}")
print()

# Test 2: SELECT * (should be blocked)
bad_sql = f"SELECT * FROM `{BQ_DATASET}.orders`"
result2 = sql_validator.validate(bad_sql)
print("Test 2 (SELECT *):")
print(f"  {result2.summary()}")
print()

# Test 3: Unauthorized table
bad_sql2 = f"SELECT col FROM `{BQ_DATASET}.secret_audit_logs`"
result3 = sql_validator.validate(bad_sql2)
print("Test 3 (unauthorized table):")
print(f"  {result3.summary()}")
print()

# Test 4: DROP statement
bad_sql3 = f"DROP TABLE `{BQ_DATASET}.orders`"
result4 = sql_validator.validate(bad_sql3)
print("Test 4 (DROP):")
print(f"  {result4.summary()}")

---
<a id="5-langgraph-nlqsql-agent"></a>
## 5. LangGraph NLQ→SQL Agent

Now we wire everything together into a **stateful agent** using LangGraph. This is the core of the tutorial.

### Why LangGraph?

LangGraph gives us:
- **Explicit state management** — Every step reads/writes to a shared state object
- **Conditional routing** — Branch based on validation results, errors, etc.
- **Built-in checkpointing** — Resume from any step; essential for HITL
- **Retry/self-correction** — Loop back to fix errors with max-attempt caps
- **Visualization** — The graph itself is a living architecture diagram

### 5.1 Agent State Definition


In [ ]:
# ============================================================
# 5.1 — Agent State Schema
# ============================================================
from typing import TypedDict, Optional, Literal, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
import operator

class NLQState(TypedDict):
    """
    State object that flows through every node in the LangGraph.
    
    Each node reads from and writes to this shared state.
    LangGraph handles state persistence and checkpointing.
    """
    # ── Input ──
    question: str                          # Original user question
    
    # ── Schema Retrieval ──
    schema_context: str                    # Formatted schema for LLM
    relevant_tables: list[str]             # Tables identified as relevant
    
    # ── SQL Generation ──
    generated_sql: str                     # Raw SQL from LLM
    sql_explanation: str                   # Plain-English explanation
    
    # ── Validation ──
    validated_sql: str                     # SQL after validation (may have LIMIT added)
    validation_result: dict                # Full validation result as dict
    is_valid: bool                         # Quick check flag
    
    # ── Execution ──
    query_results: str                     # Formatted query results
    row_count: int                         # Number of rows returned
    execution_error: str                   # Error message if execution fails
    
    # ── Control Flow ──
    attempt: int                           # Current retry attempt (max 3)
    status: str                            # PENDING, APPROVED, REJECTED, ERROR, DONE
    
    # ── Lineage ──
    lineage_log: dict                      # Full audit trail

print("✅ NLQState schema defined")

In [ ]:
# ============================================================
# 5.2 — Agent Node Functions
# ============================================================
import pandas as pd
from datetime import datetime
import traceback

# ──────────────────────────────────────────────
# NODE 1: Intent Classification & Schema Retrieval
# ──────────────────────────────────────────────
def retrieve_schema(state: NLQState) -> dict:
    """
    Classify the user's intent and retrieve relevant schema metadata.
    
    In a production system, this would use embeddings to find
    the most relevant tables. For this lab, we retrieve all tables
    since our dataset is small.
    """
    question = state["question"]
    
    # Simple keyword-based table relevance (replace with embedding similarity in production)
    relevant = []
    q_lower = question.lower()
    
    if any(w in q_lower for w in ["order", "revenue", "sales", "region", "channel"]):
        relevant.append("orders")
    if any(w in q_lower for w in ["customer", "segment", "country", "who", "buyer"]):
        relevant.append("customers")
    if any(w in q_lower for w in ["product", "category", "item", "best-selling", "sku"]):
        relevant.append("products")
    if any(w in q_lower for w in ["quantity", "discount", "line item", "unit price"]):
        relevant.append("order_items")
    
    # If no specific tables detected, include all
    if not relevant:
        relevant = list(SchemaRetriever.ALLOWED_TABLES)
    
    # Always include order_items if both orders and products are relevant (need join table)
    if "orders" in relevant and "products" in relevant and "order_items" not in relevant:
        relevant.append("order_items")
    
    schema_context = schema_retriever.get_full_context(relevant)
    
    return {
        "schema_context": schema_context,
        "relevant_tables": relevant,
        "attempt": state.get("attempt", 0),
    }


# ──────────────────────────────────────────────
# NODE 2: SQL Generation
# ──────────────────────────────────────────────
def generate_sql(state: NLQState) -> dict:
    """Generate SQL using Gemini and create a plain-English explanation."""
    question = state["question"]
    schema_context = state["schema_context"]
    attempt = state.get("attempt", 0)
    
    # If this is a retry, include the error in the prompt
    error_context = ""
    if attempt > 0 and state.get("execution_error"):
        error_context = f"""
        
## PREVIOUS ATTEMPT FAILED
The previous SQL generated an error. Fix the SQL based on this error:
Error: {state['execution_error']}
Previous SQL: {state.get('generated_sql', 'N/A')}

Generate a corrected SQL query:"""
    
    result = sql_generator.generate(
        question=question + error_context,
        schema_context=schema_context,
        dataset=BQ_DATASET,
    )
    
    # Generate plain-English explanation
    explain_prompt = f"""Explain this SQL query in one sentence for a business user.
Be concise. Do not include the SQL itself.
SQL: {result['sql']}"""
    
    from langchain_core.messages import HumanMessage
    explanation = sql_generator.llm.invoke([HumanMessage(content=explain_prompt)])
    
    return {
        "generated_sql": result["sql"],
        "sql_explanation": explanation.content.strip(),
    }


# ──────────────────────────────────────────────
# NODE 3: Validation
# ──────────────────────────────────────────────
def validate_sql(state: NLQState) -> dict:
    """Run the SQL through our multi-stage validation pipeline."""
    sql = state["generated_sql"]
    result = sql_validator.validate(sql)
    
    return {
        "validated_sql": result.sql,
        "is_valid": result.is_valid,
        "validation_result": {
            "errors": result.errors,
            "warnings": result.warnings,
            "estimated_bytes": result.estimated_bytes,
            "estimated_cost_usd": result.estimated_cost_usd,
            "summary": result.summary(),
        }
    }


# ──────────────────────────────────────────────
# NODE 4: Human Confirmation (Simulated)
# ──────────────────────────────────────────────
def request_confirmation(state: NLQState) -> dict:
    """
    Present the query to the user for confirmation.
    
    In production, this would pause the graph and wait for user input
    via LangGraph's interrupt mechanism. For this lab, we auto-approve.
    """
    vr = state["validation_result"]
    
    print("\n" + "=" * 70)
    print("🔍 CONFIRMATION REQUEST")
    print("=" * 70)
    print(f"Question:    {state['question']}")
    print(f"Explanation: {state['sql_explanation']}")
    print(f"\nGenerated SQL:\n{state['validated_sql']}")
    print(f"\nValidation:  {vr['summary']}")
    if vr.get("warnings"):
        print(f"Warnings:    {'; '.join(vr['warnings'])}")
    print("=" * 70)
    
    # Auto-approve for lab purposes
    # In production: use langgraph interrupt() here
    return {"status": "APPROVED"}


# ──────────────────────────────────────────────
# NODE 5: Execute Query
# ──────────────────────────────────────────────
def execute_query(state: NLQState) -> dict:
    """Execute the validated SQL against BigQuery and format results."""
    sql = state["validated_sql"]
    
    try:
        query_job = bq_client.query(sql)
        results = query_job.result()
        
        df = results.to_dataframe()
        row_count = len(df)
        
        if row_count == 0:
            formatted = "No results found for this query."
        else:
            formatted = df.to_string(index=False)
        
        # Build lineage log
        lineage = {
            "timestamp": datetime.now().isoformat(),
            "question": state["question"],
            "generated_sql": state["generated_sql"],
            "validated_sql": sql,
            "row_count": row_count,
            "estimated_bytes": state["validation_result"].get("estimated_bytes"),
            "estimated_cost_usd": state["validation_result"].get("estimated_cost_usd"),
            "attempt": state.get("attempt", 0) + 1,
            "status": "SUCCESS",
        }
        
        return {
            "query_results": formatted,
            "row_count": row_count,
            "execution_error": "",
            "status": "DONE",
            "lineage_log": lineage,
        }
        
    except Exception as e:
        error_msg = str(e)
        return {
            "query_results": "",
            "row_count": 0,
            "execution_error": error_msg,
            "status": "ERROR",
            "attempt": state.get("attempt", 0) + 1,
            "lineage_log": {
                "timestamp": datetime.now().isoformat(),
                "question": state["question"],
                "generated_sql": state["generated_sql"],
                "error": error_msg,
                "attempt": state.get("attempt", 0) + 1,
                "status": "ERROR",
            }
        }

print("✅ All node functions defined")

In [ ]:
# ============================================================
# 5.3 — Routing Functions (Conditional Edges)
# ============================================================

def route_after_validation(state: NLQState) -> str:
    """Route based on validation result."""
    if state["is_valid"]:
        return "request_confirmation"
    else:
        attempt = state.get("attempt", 0)
        if attempt < 3:
            return "generate_sql"       # Retry with error context
        else:
            return "max_retries"         # Give up


def route_after_execution(state: NLQState) -> str:
    """Route based on execution result."""
    if state["status"] == "DONE":
        return "done"
    elif state["status"] == "ERROR":
        attempt = state.get("attempt", 0)
        if attempt < 3:
            return "generate_sql"       # Self-correct
        else:
            return "max_retries"
    return "done"


def handle_max_retries(state: NLQState) -> dict:
    """Handle case where all retry attempts are exhausted."""
    return {
        "status": "FAILED",
        "query_results": (
            f"I was unable to generate a valid query after {state.get('attempt', 0)} attempts. "
            f"Last error: {state.get('execution_error', 'Validation failed')}\n"
            f"Please rephrase your question or contact the BI team for assistance."
        ),
        "lineage_log": {
            "timestamp": datetime.now().isoformat(),
            "question": state["question"],
            "status": "FAILED_MAX_RETRIES",
            "attempts": state.get("attempt", 0),
        }
    }

print("✅ Routing functions defined")

In [ ]:
# ============================================================
# 5.4 — Build the LangGraph
# ============================================================

workflow = StateGraph(NLQState)

# ── Add nodes ──
workflow.add_node("retrieve_schema",      retrieve_schema)
workflow.add_node("generate_sql",         generate_sql)
workflow.add_node("validate_sql",         validate_sql)
workflow.add_node("request_confirmation", request_confirmation)
workflow.add_node("execute_query",        execute_query)
workflow.add_node("max_retries",          handle_max_retries)

# ── Add edges ──
workflow.add_edge(START,                  "retrieve_schema")
workflow.add_edge("retrieve_schema",      "generate_sql")
workflow.add_edge("generate_sql",         "validate_sql")

# Conditional: after validation
workflow.add_conditional_edges(
    "validate_sql",
    route_after_validation,
    {
        "request_confirmation": "request_confirmation",
        "generate_sql":         "generate_sql",
        "max_retries":          "max_retries",
    }
)

workflow.add_edge("request_confirmation", "execute_query")

# Conditional: after execution
workflow.add_conditional_edges(
    "execute_query",
    route_after_execution,
    {
        "done":         END,
        "generate_sql": "generate_sql",
        "max_retries":  "max_retries",
    }
)

workflow.add_edge("max_retries", END)

# ── Compile ──
app = workflow.compile()
print("✅ LangGraph NLQ→SQL agent compiled")

In [ ]:
# ============================================================
# 5.5 — Visualize the Graph (Optional)
# ============================================================
# This generates a Mermaid diagram of the agent graph.
# You can paste the output into https://mermaid.live to visualize.

try:
    print(app.get_graph().draw_mermaid())
except Exception:
    print("(Graph visualization requires additional dependencies — skipping)")

In [ ]:
# ============================================================
# 5.6 — Run the LangGraph Agent
# ============================================================

def run_nlq_agent(question: str) -> dict:
    """
    Run the NLQ→SQL agent with a natural language question.
    Returns the final state with results, SQL, and lineage.
    """
    initial_state = {
        "question": question,
        "attempt": 0,
        "status": "PENDING",
    }
    
    final_state = app.invoke(initial_state)
    return final_state


# ── Test with sample questions ──
print("=" * 70)
print("🚀 Running NLQ→SQL Agent")
print("=" * 70)

result = run_nlq_agent("What is total revenue by region for completed orders?")

print("\n📊 RESULTS:")
print(result["query_results"])
print(f"\n📝 Rows returned: {result['row_count']}")
print(f"📋 Status: {result['status']}")

In [ ]:
# ============================================================
# 5.7 — More Test Queries
# ============================================================

test_queries = [
    "Show me the top 5 customers by total spend",
    "How many orders were placed per month in 2025?",
    "What is the average order value by sales channel for completed orders?",
    "Which product category generates the most revenue?",
]

for q in test_queries:
    print(f"\n{'=' * 70}")
    print(f"Q: {q}")
    print('=' * 70)
    
    result = run_nlq_agent(q)
    
    print(f"\n📊 Results:\n{result['query_results']}")
    print(f"Status: {result['status']} | Attempts: {result.get('lineage_log', {}).get('attempt', 1)}")

---
<a id="6-google-adk-nlqsql-agent"></a>
## 6. Google ADK NLQ→SQL Agent

Google's **Agent Development Kit (ADK)** provides a Gemini-optimized framework for building agents. Let's rebuild the same NLQ→SQL agent using ADK.

### Key Differences from LangGraph

| Feature | LangGraph | Google ADK |
|---------|-----------|------------|
| State management | Explicit TypedDict state | Session-based, managed by ADK |
| Tool definition | Python functions as nodes | `FunctionTool` with schemas |
| LLM integration | Via LangChain wrappers | Native Gemini integration |
| Orchestration | Graph edges & routing | Agent orchestrates tool calls |
| Deployment | Custom (Cloud Run, etc.) | Agent Engine (managed) |
| HITL | Graph interrupts | Callbacks / hooks |


In [ ]:
# ============================================================
# 6.1 — Define ADK Tools
# ============================================================
# Google ADK uses tools that the agent can call autonomously.
# Each tool is a Python function with type annotations.

from google.adk.agents import Agent
from google.adk.tools import FunctionTool
from typing import Optional

# ── Tool 1: Get Schema ──
def get_database_schema(table_names: Optional[str] = None) -> dict:
    """
    Retrieve BigQuery schema metadata for specified tables.
    
    Args:
        table_names: Comma-separated table names (e.g., 'orders,customers').
                     If not provided, returns all available tables.
    
    Returns:
        Dictionary with schema context and available tables.
    """
    if table_names:
        tables = [t.strip() for t in table_names.split(",")]
    else:
        tables = None
    
    context = schema_retriever.get_full_context(tables)
    available = list(SchemaRetriever.ALLOWED_TABLES)
    
    return {
        "schema_context": context,
        "available_tables": available,
        "note": "Use this schema to generate BigQuery Standard SQL."
    }


# ── Tool 2: Validate SQL ──
def validate_sql_query(sql: str) -> dict:
    """
    Validate a generated SQL query through safety checks and dry-run.
    
    Args:
        sql: The SQL query to validate.
    
    Returns:
        Validation result with is_valid flag, errors, warnings, and cost estimate.
    """
    result = sql_validator.validate(sql)
    return {
        "is_valid": result.is_valid,
        "validated_sql": result.sql,
        "errors": result.errors,
        "warnings": result.warnings,
        "estimated_bytes": result.estimated_bytes,
        "estimated_cost_usd": result.estimated_cost_usd,
        "summary": result.summary(),
    }


# ── Tool 3: Execute SQL ──
def execute_sql_query(sql: str) -> dict:
    """
    Execute a validated SQL query against BigQuery.
    Only call this AFTER validate_sql_query returns is_valid=True.
    
    Args:
        sql: The validated SQL query to execute.
    
    Returns:
        Query results as formatted text, with row count.
    """
    try:
        query_job = bq_client.query(sql)
        df = query_job.result().to_dataframe()
        
        if len(df) == 0:
            return {"results": "No results found.", "row_count": 0, "status": "SUCCESS"}
        
        return {
            "results": df.to_string(index=False),
            "row_count": len(df),
            "status": "SUCCESS",
        }
    except Exception as e:
        return {
            "results": "",
            "row_count": 0,
            "status": "ERROR",
            "error": str(e),
        }


# ── Tool 4: Get Sample Data ──
def get_sample_data(table_name: str, limit: int = 3) -> dict:
    """
    Get sample rows from a table to understand data format.
    
    Args:
        table_name: Name of the table to sample.
        limit: Number of rows to return (default 3, max 10).
    
    Returns:
        Sample rows formatted as text.
    """
    limit = min(limit, 10)
    sample = schema_retriever.get_sample_rows(table_name, limit)
    return {"sample_data": sample}


print("✅ ADK tools defined: get_database_schema, validate_sql_query, execute_sql_query, get_sample_data")

In [ ]:
# ============================================================
# 6.2 — Create ADK Agent
# ============================================================

ADK_SYSTEM_INSTRUCTION = f"""You are an expert BigQuery SQL analyst agent. Your job is to help users
query a BigQuery data warehouse using natural language.

## YOUR WORKFLOW (follow these steps IN ORDER):
1. FIRST, call get_database_schema to understand the available tables and columns.
2. Generate a BigQuery Standard SQL query based on the user's question and the schema.
3. Call validate_sql_query with your generated SQL to check for errors and estimate cost.
4. If validation fails, fix the SQL based on the error messages and re-validate (max 3 attempts).
5. If validation passes, present the SQL and explanation to the user.
6. Call execute_sql_query to run the validated query.
7. Present the results in a clear, readable format.

## SQL RULES:
- Use BigQuery Standard SQL syntax.
- NEVER use SELECT * — always specify columns.
- For revenue/sales, only include orders with status = 'COMPLETED'.
- Use table aliases (o for orders, c for customers, p for products, oi for order_items).
- Use fully qualified table names: `{BQ_DATASET}.table_name`
- Always include appropriate WHERE, GROUP BY, and ORDER BY clauses.
- Today's date: use CURRENT_DATE() for relative date references.

## SAFETY:
- Only query tables returned by get_database_schema.
- Always validate before executing.
- If you cannot answer the question with available data, explain why.
"""

# Create the agent
nlq_agent_adk = Agent(
    name="nlq_sql_agent",
    model="gemini-2.5-flash",
    description="Natural language to BigQuery SQL agent",
    instruction=ADK_SYSTEM_INSTRUCTION,
    tools=[
        get_database_schema,
        validate_sql_query,
        execute_sql_query,
        get_sample_data,
    ],
)

print("✅ Google ADK NLQ→SQL Agent created")
print(f"   Model: gemini-2.5-flash")
print(f"   Tools: {[t.__name__ if callable(t) else t.name for t in nlq_agent_adk.tools]}")

In [ ]:
# ============================================================
# 6.3 — Run ADK Agent
# ============================================================
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Create session service and runner
session_service = InMemorySessionService()
runner = Runner(
    agent=nlq_agent_adk,
    app_name="nlq_sql_app",
    session_service=session_service,
)

async def run_adk_agent(question: str, user_id: str = "lab_user"):
    """Run the ADK agent with a natural language question."""
    # Create or get session
    session = await session_service.create_session(
        app_name="nlq_sql_app",
        user_id=user_id,
    )
    
    # Create user message
    user_message = types.Content(
        role="user",
        parts=[types.Part(text=question)],
    )
    
    # Run agent and collect responses
    final_response = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=user_message,
    ):
        if event.is_final_response():
            for part in event.content.parts:
                if part.text:
                    final_response += part.text
    
    return final_response


# ── Run a test query ──
import asyncio

question = "What is total revenue by region for completed orders?"
print(f"Q: {question}")
print("=" * 70)

response = await run_adk_agent(question)
print(response)

In [ ]:
# ============================================================
# 6.4 — Test ADK Agent with Multiple Queries
# ============================================================

adk_test_queries = [
    "Who are the top 3 customers by total spend?",
    "How many orders per month in 2025?",
    "What is the best-selling product category by revenue?",
]

for q in adk_test_queries:
    print(f"\n{'=' * 70}")
    print(f"Q: {q}")
    print('=' * 70)
    response = await run_adk_agent(q)
    print(response)

---
<a id="7-human-in-the-loop"></a>
## 7. Human-in-the-Loop (HITL)

In production, you **never** auto-execute SQL against sensitive data. LangGraph provides a built-in **interrupt** mechanism that pauses the graph and waits for human approval.

### 7.1 LangGraph with Interrupt


In [ ]:
# ============================================================
# 7.1 — LangGraph HITL with Checkpointing
# ============================================================
from langgraph.checkpoint.memory import MemorySaver

# Create a checkpointer for state persistence
checkpointer = MemorySaver()

# Rebuild the graph with an interrupt before execution
hitl_workflow = StateGraph(NLQState)

hitl_workflow.add_node("retrieve_schema",      retrieve_schema)
hitl_workflow.add_node("generate_sql",         generate_sql)
hitl_workflow.add_node("validate_sql",         validate_sql)
hitl_workflow.add_node("request_confirmation", request_confirmation)
hitl_workflow.add_node("execute_query",        execute_query)
hitl_workflow.add_node("max_retries",          handle_max_retries)

hitl_workflow.add_edge(START,                  "retrieve_schema")
hitl_workflow.add_edge("retrieve_schema",      "generate_sql")
hitl_workflow.add_edge("generate_sql",         "validate_sql")

hitl_workflow.add_conditional_edges(
    "validate_sql",
    route_after_validation,
    {
        "request_confirmation": "request_confirmation",
        "generate_sql":         "generate_sql",
        "max_retries":          "max_retries",
    }
)

hitl_workflow.add_edge("request_confirmation", "execute_query")

hitl_workflow.add_conditional_edges(
    "execute_query",
    route_after_execution,
    {"done": END, "generate_sql": "generate_sql", "max_retries": "max_retries"}
)

hitl_workflow.add_edge("max_retries", END)

# Compile WITH interrupt before execute_query
hitl_app = hitl_workflow.compile(
    checkpointer=checkpointer,
    interrupt_before=["execute_query"],  # Pause here for human approval
)

print("✅ HITL agent compiled with interrupt before execute_query")

In [ ]:
# ============================================================
# 7.2 — Run HITL Agent (Pause → Approve → Resume)
# ============================================================

# Step 1: Run until interrupt
config = {"configurable": {"thread_id": "hitl-demo-1"}}

initial_state = {
    "question": "What is total revenue by region?",
    "attempt": 0,
    "status": "PENDING",
}

print("Step 1: Running agent until confirmation point...")
print("=" * 70)

# This will run and PAUSE before execute_query
for event in hitl_app.stream(initial_state, config=config):
    # Print progress
    for node_name, node_output in event.items():
        if node_name != "__end__":
            print(f"  ✅ Completed: {node_name}")

# Check current state
current_state = hitl_app.get_state(config)
print(f"\n⏸️  Agent paused. Next step: {current_state.next}")
print(f"   SQL: {current_state.values.get('validated_sql', 'N/A')[:100]}...")

In [ ]:
# ============================================================
# 7.3 — Approve and Resume
# ============================================================

# In production, this is where a human reviews the SQL
# and clicks "Approve" or "Reject" in the UI.

print("Step 2: Human approves → Resuming execution...")
print("=" * 70)

# Resume from checkpoint (pass None to continue from where we left off)
for event in hitl_app.stream(None, config=config):
    for node_name, node_output in event.items():
        if node_name != "__end__":
            print(f"  ✅ Completed: {node_name}")

# Get final results
final = hitl_app.get_state(config)
print(f"\n📊 Results:\n{final.values.get('query_results', 'N/A')}")
print(f"Status: {final.values.get('status', 'N/A')}")

---
<a id="8-self-correction-loop"></a>
## 8. Self-Correction Loop

The agent handles errors by feeding the error message back into the SQL generation step. This is the ReAct pattern in action:

```
Generate SQL → Validate → Execute → ERROR!
     ↑                                  │
     └──── Feed error into prompt ──────┘
                (max 3 attempts)
```

The retry logic is already built into our LangGraph routing (Section 5.3). Let's test it with a deliberately tricky query.


In [ ]:
# ============================================================
# 8.1 — Test Self-Correction
# ============================================================

# This query is intentionally tricky — it asks about "profit margin"
# which requires computing (price - cost) / price, not a direct column.
tricky_question = "What is the profit margin percentage for each product category?"

print(f"Q: {tricky_question}")
print("=" * 70)
print("Watch the agent potentially self-correct if the first attempt fails...\n")

result = run_nlq_agent(tricky_question)

print(f"\n📊 Results:\n{result['query_results']}")
print(f"\nFinal SQL:\n{result.get('validated_sql', result.get('generated_sql', 'N/A'))}")
print(f"\nAttempts: {result.get('lineage_log', {}).get('attempt', 'N/A')}")
print(f"Status: {result['status']}")

---
<a id="9-caching--lineage-logging"></a>
## 9. Caching & Lineage Logging

### 9.1 Multi-Layer Caching

Production NLQ→SQL systems use caching at every layer to reduce cost and latency:

| Layer | What's Cached | TTL |
|-------|--------------|-----|
| **Schema Cache** | Table metadata, column descriptions | 1-6 hours |
| **Plan Cache** | Question → SQL mapping | 24 hours |
| **Result Cache** | Query results | 5-60 minutes |
| **Embedding Cache** | Question embeddings | Persistent |


In [ ]:
# ============================================================
# 9.1 — Plan Cache (Question → SQL)
# ============================================================
import hashlib
from datetime import datetime, timedelta

class PlanCache:
    """
    Caches NLQ→SQL mappings to avoid regenerating SQL for similar questions.
    
    Uses a simple in-memory dict. In production, use Redis or Firestore
    with TTL-based expiration.
    """
    
    def __init__(self, ttl_hours: int = 24):
        self._cache: dict[str, dict] = {}
        self._ttl = timedelta(hours=ttl_hours)
    
    def _normalize_question(self, question: str) -> str:
        """Normalize question for cache key (lowercase, strip whitespace)."""
        return " ".join(question.lower().strip().split())
    
    def _make_key(self, question: str) -> str:
        normalized = self._normalize_question(question)
        return hashlib.sha256(normalized.encode()).hexdigest()[:16]
    
    def get(self, question: str) -> dict | None:
        """Look up cached SQL for a question. Returns None on cache miss."""
        key = self._make_key(question)
        entry = self._cache.get(key)
        
        if entry is None:
            return None
        
        # Check TTL
        if datetime.now() - entry["cached_at"] > self._ttl:
            del self._cache[key]
            return None
        
        entry["cache_hits"] = entry.get("cache_hits", 0) + 1
        return entry
    
    def put(self, question: str, sql: str, validation_result: dict):
        """Store a successful NLQ→SQL mapping."""
        key = self._make_key(question)
        self._cache[key] = {
            "question": question,
            "sql": sql,
            "validation_result": validation_result,
            "cached_at": datetime.now(),
            "cache_hits": 0,
        }
    
    def stats(self) -> dict:
        """Return cache statistics."""
        return {
            "entries": len(self._cache),
            "total_hits": sum(e.get("cache_hits", 0) for e in self._cache.values()),
        }

# ── Instantiate ──
plan_cache = PlanCache(ttl_hours=24)
print("✅ PlanCache initialized (TTL: 24h)")

In [ ]:
# ============================================================
# 9.2 — Lineage Logger
# ============================================================
import json

class LineageLogger:
    """
    Logs every NLQ→SQL translation for audit, debugging, and evaluation.
    
    In production, write to BigQuery or Cloud Logging.
    For this lab, we use an in-memory list.
    """
    
    def __init__(self):
        self._logs: list[dict] = []
    
    def log(self, entry: dict):
        """Add a lineage entry."""
        entry["log_id"] = len(self._logs) + 1
        entry["logged_at"] = datetime.now().isoformat()
        self._logs.append(entry)
    
    def get_logs(self, limit: int = 10) -> list[dict]:
        """Get recent log entries."""
        return self._logs[-limit:]
    
    def to_dataframe(self):
        """Convert logs to a pandas DataFrame for analysis."""
        import pandas as pd
        return pd.DataFrame(self._logs)
    
    def export_to_bigquery(self, table_id: str):
        """
        Export logs to BigQuery for persistent storage.
        Call this periodically or at end of session.
        """
        if not self._logs:
            print("No logs to export.")
            return
        
        import pandas as pd
        df = pd.DataFrame(self._logs)
        
        # Write to BigQuery
        job_config = bigquery.LoadJobConfig(
            write_disposition="WRITE_APPEND",
            autodetect=True,
        )
        
        try:
            job = bq_client.load_table_from_dataframe(
                df, table_id, job_config=job_config
            )
            job.result()
            print(f"✅ Exported {len(self._logs)} log entries to {table_id}")
        except Exception as e:
            print(f"⚠️ Export failed: {e}")

# ── Instantiate ──
lineage_logger = LineageLogger()
print("✅ LineageLogger initialized")

In [ ]:
# ============================================================
# 9.3 — Integrated Agent Run with Caching + Logging
# ============================================================

def run_nlq_agent_with_cache(question: str) -> dict:
    """
    Run the NLQ→SQL agent with plan caching and lineage logging.
    """
    # Check cache first
    cached = plan_cache.get(question)
    if cached:
        print(f"⚡ CACHE HIT for: {question[:50]}...")
        lineage_logger.log({
            "question": question,
            "sql": cached["sql"],
            "source": "cache",
            "status": "CACHE_HIT",
        })
        
        # Re-execute the cached SQL (results may have changed)
        try:
            df = bq_client.query(cached["sql"]).result().to_dataframe()
            return {
                "query_results": df.to_string(index=False),
                "row_count": len(df),
                "status": "DONE",
                "source": "cache",
                "sql": cached["sql"],
            }
        except Exception as e:
            print(f"  Cache hit but execution failed, regenerating: {e}")
    
    # Cache miss → run full agent
    print(f"🔄 Cache miss, running agent for: {question[:50]}...")
    result = run_nlq_agent(question)
    
    # Cache the successful result
    if result.get("status") == "DONE":
        plan_cache.put(
            question,
            result.get("validated_sql", result.get("generated_sql", "")),
            result.get("validation_result", {}),
        )
    
    # Log the result
    lineage_logger.log(result.get("lineage_log", {"question": question, "status": result.get("status")}))
    
    return result


# ── Test: First call (cache miss) ──
print("--- Call 1 (cache miss expected) ---")
r1 = run_nlq_agent_with_cache("What is total revenue by region?")
print(f"Results: {r1['query_results'][:200]}")

print("\n--- Call 2 (cache hit expected) ---")
r2 = run_nlq_agent_with_cache("What is total revenue by region?")
print(f"Results: {r2['query_results'][:200]}")

print(f"\nCache stats: {plan_cache.stats()}")

---
<a id="10-evaluation--testing"></a>
## 10. Evaluation & Testing

### Golden Test Set

A production NLQ→SQL system needs a **golden test set** — a curated list of questions with known-correct SQL and expected results. We evaluate:

1. **SQL Correctness** — Does the query parse and execute without errors?
2. **Result Accuracy** — Do results match the golden answers?
3. **Latency** — End-to-end time from question to results
4. **Cost** — Bytes processed per query


In [ ]:
# ============================================================
# 10.1 — Golden Test Set
# ============================================================
import time

GOLDEN_TEST_SET = [
    {
        "question": "What is total revenue by region?",
        "expected_columns": ["region", "total_revenue"],
        "expected_min_rows": 1,
        "must_contain": ["North America"],  # We know this region has data
    },
    {
        "question": "How many completed orders are there?",
        "expected_columns": ["order_count"],  # or similar
        "expected_min_rows": 1,
        "must_contain": [],
    },
    {
        "question": "Who are the top 3 customers by total spend?",
        "expected_columns": ["name"],
        "expected_min_rows": 1,
        "must_contain": ["Acme Corp"],  # Known top customer
    },
    {
        "question": "What is the average order value by sales channel?",
        "expected_columns": ["channel"],
        "expected_min_rows": 1,
        "must_contain": ["web"],
    },
    {
        "question": "Show me monthly order counts for 2025",
        "expected_columns": ["order_month", "order_count"],
        "expected_min_rows": 1,
        "must_contain": [],
    },
    {
        "question": "Which product has the highest total quantity sold?",
        "expected_columns": ["name"],
        "expected_min_rows": 1,
        "must_contain": [],
    },
    {
        "question": "What is total revenue for Enterprise customers?",
        "expected_columns": ["total_revenue"],
        "expected_min_rows": 1,
        "must_contain": [],
    },
    {
        "question": "List all cancelled or refunded orders",
        "expected_columns": ["order_id", "status"],
        "expected_min_rows": 1,
        "must_contain": ["CANCELLED"],
    },
]

print(f"✅ Golden test set: {len(GOLDEN_TEST_SET)} queries")

In [ ]:
# ============================================================
# 10.2 — Run Evaluation
# ============================================================
import pandas as pd

eval_results = []

for i, test in enumerate(GOLDEN_TEST_SET):
    print(f"\n[{i+1}/{len(GOLDEN_TEST_SET)}] {test['question'][:60]}...")
    
    start_time = time.time()
    
    try:
        result = run_nlq_agent(test["question"])
        elapsed = time.time() - start_time
        
        # Check: did it succeed?
        succeeded = result.get("status") == "DONE"
        
        # Check: does result contain expected content?
        results_text = result.get("query_results", "")
        content_match = all(
            expected.lower() in results_text.lower()
            for expected in test["must_contain"]
        ) if test["must_contain"] else True
        
        # Check: row count
        row_ok = result.get("row_count", 0) >= test["expected_min_rows"]
        
        eval_results.append({
            "question": test["question"][:60],
            "status": "PASS" if (succeeded and content_match and row_ok) else "FAIL",
            "executed": succeeded,
            "content_match": content_match,
            "row_count": result.get("row_count", 0),
            "latency_s": round(elapsed, 2),
            "sql": result.get("validated_sql", result.get("generated_sql", "N/A"))[:80],
        })
        
        status = "✅" if (succeeded and content_match and row_ok) else "❌"
        print(f"  {status} Status={result.get('status')} | Rows={result.get('row_count', 0)} | {elapsed:.1f}s")
        
    except Exception as e:
        elapsed = time.time() - start_time
        eval_results.append({
            "question": test["question"][:60],
            "status": "ERROR",
            "executed": False,
            "content_match": False,
            "row_count": 0,
            "latency_s": round(elapsed, 2),
            "sql": str(e)[:80],
        })
        print(f"  ❌ Error: {e}")

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# 10.3 — Evaluation Report
# ============================================================

eval_df = pd.DataFrame(eval_results)

# Summary metrics
total = len(eval_df)
passed = len(eval_df[eval_df["status"] == "PASS"])
accuracy = passed / total * 100
avg_latency = eval_df["latency_s"].mean()
p95_latency = eval_df["latency_s"].quantile(0.95)

print("📊 EVALUATION REPORT")
print("=" * 70)
print(f"Total queries:    {total}")
print(f"Passed:           {passed}/{total} ({accuracy:.0f}%)")
print(f"Avg latency:      {avg_latency:.1f}s")
print(f"P95 latency:      {p95_latency:.1f}s")
print()

# Detailed results table
print(eval_df[["question", "status", "row_count", "latency_s"]].to_string(index=False))

# SLO check
print("\n--- SLO Check ---")
print(f"Accuracy ≥ 85%:  {'✅ PASS' if accuracy >= 85 else '❌ FAIL'} ({accuracy:.0f}%)")
print(f"P95 latency < 5s: {'✅ PASS' if p95_latency < 5 else '❌ FAIL'} ({p95_latency:.1f}s)")

---
<a id="11-langgraph-vs-adk-comparison"></a>
## 11. LangGraph vs Google ADK — When to Use Which

### Decision Framework

| Criterion | LangGraph | Google ADK |
|-----------|-----------|------------|
| **Control over flow** | Full control — you define every edge and condition | Agent decides tool call order (LLM-driven) |
| **State management** | Explicit state schema with checkpointing | Session-based, managed by runtime |
| **HITL support** | Built-in interrupt/resume with checkpoints | Callback-based hooks |
| **Multi-model support** | Any LLM via LangChain wrappers | Gemini-optimized (primary), others via plugins |
| **Deployment** | Cloud Run, Cloud Functions (DIY) | Agent Engine (managed by Google) |
| **Observability** | Via LangSmith, Langfuse, Cloud Trace | Native Agent Engine monitoring |
| **Self-correction** | Explicit retry edges in graph | LLM naturally retries via tool calling |
| **Multi-agent** | LangGraph subgraphs | ADK multi-agent with A2A protocol |
| **Vendor lock-in** | Low (framework-agnostic) | Medium (Google ecosystem) |

### Recommendation

| Use Case | Recommended |
|----------|-------------|
| Need fine-grained control over every step | **LangGraph** |
| Need managed deployment with zero infra | **Google ADK + Agent Engine** |
| Multi-cloud or multi-LLM strategy | **LangGraph** |
| All-in on Google Cloud ecosystem | **Google ADK** |
| Complex multi-agent workflows with A2A | **Google ADK** |
| Need to integrate with existing LangChain code | **LangGraph** |
| Rapid prototyping with Gemini | **Google ADK** |

### Key Takeaway

Both frameworks can build the same NLQ→SQL agent. The choice depends on your team's ecosystem preference and operational requirements. In many production systems, teams use **LangGraph for orchestration logic** and **ADK tools for Google Cloud integration** — they're not mutually exclusive.


---
## 🎯 Summary & Next Steps

### What We Built

In this notebook, you implemented a **production-grade NLQ→SQL Workflow Agent** with:

1. **Schema Retriever** — Metadata-aware retrieval from BigQuery INFORMATION_SCHEMA with allow-lists and PII protection
2. **SQL Generator** — Gemini-powered SQL generation with optimized prompts, few-shot examples, and output constraints
3. **Validation Pipeline** — Four-stage safety gates (policy, allow-list, dry-run, row-limit)
4. **LangGraph Agent** — Stateful graph with conditional routing, self-correction, and checkpointing
5. **Google ADK Agent** — Same capabilities using Google's native agent framework
6. **Human-in-the-Loop** — Interrupt-based confirmation before execution
7. **Caching & Lineage** — Plan cache, result cache, and full audit trail
8. **Evaluation** — Golden test set with accuracy and latency SLOs

### Production Checklist

Before deploying to production, ensure:

- [ ] Replace in-memory caches with Redis or Firestore
- [ ] Add embedding-based table relevance (not keyword matching)
- [ ] Deploy to Cloud Run or Agent Engine
- [ ] Wire lineage logging to BigQuery or Cloud Logging
- [ ] Set up Cloud Trace for end-to-end latency monitoring
- [ ] Configure VPC-SC perimeter around BigQuery + Vertex AI
- [ ] Use CMEK for prompt/cache encryption
- [ ] Build SLO dashboard in Cloud Monitoring
- [ ] Add Langfuse for LLM-specific observability
- [ ] Run red-team testing for prompt injection attacks

### Assignment-1 Reminder

**Due Day 9:** PromptOps app + structured outputs + ≥2 tools

---

*Generative AI Architect Program — Batch 7 | Blue Data Consulting | April 2026*
